In [ ]:
import pandas as pd
import numpy as np
import os
import re

In [ ]:
ESG_files = {}

In [ ]:
US_dollar_rate = 85.33

# Removing unnecessary caharacters and making units uniform

def clean_value(value):
  if type(value) == str:
    value = value.replace(',', '')
    if value.startswith('INR '):
      return float(value.replace('INR ', ''))
    elif value.startswith('$'):
      usd = float(value.replace('$', ''))
      return usd * US_dollar_rate
    elif value.endswith('%'):
      try:
        return float(value.replace('%', ''))/100
      except ValueError:
        return value
    elif value.endswith(' Yrs'):
      return float(value.replace(' Yrs', ''))

  elif type(value) == int:
    return float(value)

  return value

In [ ]:
# Converting the data in suitable format which can be used for dashboard building

def proper_format(df):
  df.drop(df.columns[0], axis=1, inplace=True)
  df.drop(index=[0,1,2,3], axis = 0, inplace = True)
  df.columns = df.iloc[0].astype(str).str[:4]
  df.columns.name = None
  df.rename(columns = {'Peri':'Parameter name'}, inplace = True)
  df = df[1:].reset_index(drop=True)

  df.drop(index=range(0,6), axis = 0, inplace = True)
  df['Parameter name'] = df['Parameter name'].str.replace(r"\s*\(.*?\)", "", regex=True).str.strip()


  df.dropna(axis=0, subset = df.columns[1:],how='all', inplace=True)

  df.replace('--', np.nan, inplace=True)
  df.dropna(axis=1, how='all', inplace=True)


  df = df.melt(id_vars = ['Parameter name'], var_name = 'Year', value_name = 'Value')

  df['Year'] = pd.to_datetime(df['Year'], format='%Y', errors='coerce').dt.year

  df['Value'] = df['Value'].apply(clean_value)


  df.replace(True, 1, inplace=True)
  df.replace(False, 0, inplace=True)

  print(df['Value'][df['Value'].apply(lambda x: isinstance(x, str))].unique())


  df = df.reset_index(drop = True)

  return df

In [ ]:
def load_files(folder_path, save_path):
  files = [f for f in os.listdir(folder_path) if f.endswith('.xlsx')]

  subfolders = {'Environment': os.path.join(save_path, 'Environment'),
                'Social': os.path.join(save_path, 'Social'),
                'Governance': os.path.join(save_path, 'Governance'),
                'Controversies': os.path.join(save_path, 'Controversies') }

  for path in subfolders.values():
    os.makedirs(path, exist_ok = True)

  for file in files:
    file_path = os.path.join(folder_path, file)

    df = pd.ExcelFile(file_path)

    dfs = {sheet_name: proper_format(df.parse(sheet_name)) for sheet_name in df.sheet_names}

    stock_name = file.replace('.xlsx', '').strip()

    ESG_files[stock_name] = dfs

    for sheet_name, sheet_data in dfs.items():

      sheet_folder = subfolders.get(sheet_name, save_path)
      save_sheet_path = os.path.join(sheet_folder, f"{stock_name}_{sheet_name}.csv")

      sheet_data.to_csv(save_sheet_path, index=False)

      print(f"✅ {sheet_name} sheet from {file} saved at {save_sheet_path}")

    save_file_path = os.path.join(save_path, f"{file}.csv")
    with open(save_file_path, 'w', encoding = 'utf-8') as f:
      for sheet, data in dfs.items():
        f.write(f"{sheet_name} \n")
        data.to_csv(f, index=False)
        f.write('\n\n')

    print(f"✅ {file} loaded and saved as {save_file_path}")





    print(f"✅ {files} loaded")


In [ ]:
folder_path = '/content/drive/MyDrive/IIMA_Internship/Energy_Sector/Data/ESG_data/Raw_data'
save_path = '/content/drive/MyDrive/IIMA_Internship/Energy_Sector/Data/ESG_data/Preprocessed'


load_files(folder_path, save_path)

<ipython-input-4-40f3b607d60c>:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('--', np.nan, inplace=True)


NameError: name 'clean_value' is not defined

100.00%
97.46%
99.44%
99.96%


1